In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob

from processing import *
from wavelengths import *
from fit_prefilter import *
#from fit_voigt import *

In [2]:
dark_file = '/home/ulyanov/data/solo/phi/dark/solo_CAL1_phi-fdt-dark_20240205T033810_V202402220119C_0422051001.fits.gz'

folder = '/home/ulyanov/data/solo/phi/prefilter/calibration/2025-09-16/'
#folder = '/home/ulyanov/data/solo/phi/prefilter/calibration/2024-07-10/'
files = sorted(glob.glob(folder + '*.fits.gz'))

#for file in files:
#    process(file, dark_file=dark_file,
#            _calc_wavelengths=True, _find_center=True, _mask=True, _rebin=8, to_file=True)

In [3]:
files = sorted(glob.glob('*.fits'))

In [95]:
k_T = 0.035

wvlns = []
temperatures = []
datas = []
mus = []

for file in files:
    with fits.open(file) as hdul:
        data = hdul[0].data
        header = hdul[0].header

    nx, ny = header['NAXIS2'], header['NAXIS1']
    xc, yc = header['CRPIX2'] - 1, header['CRPIX1'] - 1
    rsun = header['RSUN_ARC'] / header['CDELT1']

    xi, yi = np.mgrid[:nx,:ny]
    mu = np.sqrt((rsun ** 2 - (xi - xc) ** 2 - (yi - yc) ** 2).clip(0)) / rsun

    datas += [data]
    wvlns += [read_wavelengths(header)]
    temperatures += [header['FGOV1PT1']]
    mus += [mu]

datas = np.swapaxes(np.concat(datas, axis=1), 0, 1)
wvlns = np.array(wvlns)
temperatures = np.array(temperatures)
mus = np.array(mus)

wvlns -= k_T * (np.expand_dims(temperatures, -1) - 61)

In [96]:
x, y = 50, 200

plt.figure(figsize=(10,8))
plt.plot(wvlns[0], datas[0,:,x,y])
plt.plot(wvlns[1], datas[1,:,x,y])
plt.plot(wvlns[2], datas[2,:,x,y])
plt.tight_layout()

In [97]:
dwv = 0.01

a, b = np.max(np.min(wvlns, axis=-1)), np.min(np.max(wvlns, axis=-1))
a = np.round(a + dwv / 2, 2)
b = np.round(b - dwv / 2, 2)

wv = np.arange(a, b + dwv / 2, dwv)

In [98]:
from fitting import interpolate

Q = []
for i in range(3):
    Q += [interpolate(datas[i], wvlns[i], np.expand_dims(wv, (-1,-2)))]
Q = np.array(Q)

In [99]:
x, y = 50, 200

plt.figure(figsize=(10,8))
for i in range(3):
    plt.plot(wv, Q[i,:,x,y])
plt.tight_layout()

In [100]:
params, gamma, res = fit_prefilter(Q[1,:], Q[2,:], wv, np.expand_dims(mus[0], 0),
                   axis=0, lam=0, niter=5)

print(gamma, np.nanmedian(res))

[ 0.05862825 -0.014835    0.04899012  0.35882016] 66.37404733646216


In [101]:
params1 = np.expand_dims(params, 1)
params2 = params1.copy()
params2[0] += gamma[-1]

Q1 = datas[1] / voigt_func((np.expand_dims(wvlns[1], (1,2)), np.expand_dims(mus[0], 0), 0), *params1, *gamma)[0]
Q2 = datas[2] / voigt_func((np.expand_dims(wvlns[2], (1,2)), np.expand_dims(mus[0], 0), 0), *params2, *gamma)[0]

In [105]:
#x, y = 50, 200
x, y = 70, 60

plt.figure(figsize=(10,8))
plt.plot(wvlns[1], datas[1,:,x,y])
plt.plot(wvlns[2], datas[2,:,x,y])
plt.plot(wvlns[1], Q1[:,x,y])
plt.plot(wvlns[2], Q2[:,x,y])
plt.tight_layout()

In [103]:
plt.figure(figsize=(10,10))
plt.imshow(params[0], 'seismic', vmin=np.min(wv), vmax=np.max(wv))
plt.tight_layout()

In [104]:
plt.figure(figsize=(10,10))
plt.imshow(-params[1], 'inferno', vmin=0.05, vmax=0.08)
plt.tight_layout()

In [86]:
plt.figure(figsize=(10,10))
plt.imshow(res, 'inferno', vmin=0, vmax=200)
plt.tight_layout()

In [64]:
plt.figure(figsize=(10,10))
plt.imshow(Q2[-1] / Q2[0], 'inferno', vmin=0.8, vmax=1.05)
plt.tight_layout()

In [66]:
plt.figure(figsize=(10,10))
plt.imshow(Q1[-1] / Q1[0], 'inferno', vmin=0.65, vmax=1.05)
plt.tight_layout()